# Unidad 4 · Cuaderno 02 · Incertidumbre y consistencia

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2

**Unidad 4.** Validación, interpretación y comunicación de resultados
· **Subtema del plan 4.2**

Este cuaderno ejecuta lo que el libro expone en la sección 4.4. El texto no
repite la teoría, remite a ella por número de definición, de teorema, de
ejemplo, de listado, de tabla o de figura, y se ocupa de reproducir los
resultados publicados y de verificarlos.

**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad4/U4_02_incertidumbre_y_consistencia.ipynb)

## Objetivos de aprendizaje

1. Clasificar la incertidumbre de una predicción como aleatoria o epistémica, según las Definiciones 4.8 y 4.9 del libro.
2. Propagar incertidumbres por la ley de primer orden del Teorema 4.4 y construir el presupuesto de la Definición 4.10.
3. Propagar distribuciones por Monte Carlo y decidir con el Algoritmo 4.4 cuándo la linealización deja de ser adecuada.
4. Aplicar las cuatro comprobaciones de consistencia que la sección 4.4.2 recomienda antes de propagar cualquier incertidumbre.
5. Redondear un resultado y su incertidumbre con cifras significativas coherentes, y declarar por escrito las cinco limitaciones.

## Puesta a punto

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("Entorno listo. Colab:", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
})

# Bandera de los ejercicios guiados. En la versión de trabajo vale False
# para que el cuaderno corra completo aunque falten celdas por resolver.
REVISAR = False


def verificar(nombre: str, obtenido, esperado: float,
              tol: float = 1.0e-3) -> bool:
    """Compara un resultado con el valor esperado sin detener el cuaderno."""
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[pendiente] {nombre}, la celda marcada COMPLETE sigue sin resolver")
        return False
    escala = abs(esperado) if esperado != 0.0 else 1.0
    error = abs(float(obtenido) - esperado) / escala
    estado = "ok" if error <= tol else "revisar"
    print(f"[{estado}] {nombre}, obtenido {float(obtenido):.6g}, "
          f"esperado {esperado:.6g}, error relativo {error:.2e}")
    if REVISAR:
        assert error <= tol, f"{nombre} no coincide con el valor esperado"
    return error <= tol


print("Semilla del curso:", SEMILLA)

In [ ]:
# Acceso a datos/ que funciona en Colab y en local, sin rutas absolutas.
# Si la carpeta no viaja con el cuaderno, las series se reconstruyen con la
# semilla del curso y con las cifras que el libro publica.

def carpeta_datos() -> Path:
    """Ubica datos/ subiendo por el árbol, o la crea junto al cuaderno."""
    base = Path.cwd()
    for nivel in [base, *base.parents][:4]:
        for candidata in (nivel / "datos", nivel / "03_cuadernos" / "datos"):
            if candidata.is_dir():
                return candidata
    destino = base / "datos"
    destino.mkdir(parents=True, exist_ok=True)
    return destino


def _cinetica_monod() -> pd.DataFrame:
    return pd.DataFrame({
        "S_g_L": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0, 35.0],
        "mu_1_h": [0.0702, 0.1248, 0.1919, 0.2496, 0.2761,
                   0.3205, 0.3689, 0.3709, 0.3770, 0.3773]})


def _secado_calibracion() -> pd.DataFrame:
    t = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 7.0, 8.0])
    gen = np.random.default_rng(SEMILLA)
    mr = np.round(np.exp(-0.350 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _secado_validacion() -> pd.DataFrame:
    t = np.array([0.5, 1.0, 1.5, 2.0, 2.75, 3.5, 4.5, 5.5, 6.5, 8.0])
    gen = np.random.default_rng(SEMILLA + 1)
    mr = np.round(np.exp(-0.362 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _caudal_mensual() -> pd.DataFrame:
    obs = [6.21, 5.01, 4.30, 9.39, 14.85, 18.28, 16.85, 17.12, 21.51, 22.75,
           19.33, 12.13, 7.65, 6.14, 4.54, 8.58, 14.73, 21.82, 14.36, 17.57,
           24.54, 31.26, 21.30, 11.27, 9.71, 6.09, 5.39, 10.86, 23.94, 20.68,
           19.39, 22.16, 32.76, 38.82, 21.33, 12.99, 5.97, 5.83, 5.07, 8.20,
           18.23, 18.93, 12.72, 16.63, 22.38, 27.76, 18.00, 12.27]
    sim = [7.70, 4.70, 4.43, 7.83, 14.55, 16.00, 15.33, 17.97, 20.37, 24.45,
           17.19, 13.27, 10.40, 6.93, 5.40, 10.19, 16.72, 22.35, 13.78, 17.44,
           24.20, 29.67, 17.56, 13.20, 12.94, 2.59, 5.60, 12.74, 21.94, 16.05,
           18.74, 17.99, 27.76, 24.48, 15.41, 12.64, 6.83, 6.11, 5.24, 11.99,
           19.88, 20.51, 12.28, 17.89, 19.38, 19.28, 14.48, 8.00]
    return pd.DataFrame({"mes": np.arange(1, 49),
                         "periodo": ["calibracion"] * 24 + ["validacion"] * 24,
                         "Q_obs_m3_s": obs, "Q_sim_m3_s": sim})


def _arreglo_fotovoltaico() -> pd.DataFrame:
    return pd.DataFrame({"configuracion": ["Base", "Optima", "Sombreado"],
                         "H_kWh_m2": [1980.0, 2035.0, 1910.0],
                         "u_rel_H": [0.04, 0.04, 0.04]})


def _entradas_vertedero() -> pd.DataFrame:
    return pd.DataFrame({"magnitud": ["C_d", "b", "h"],
                         "unidad": ["1", "m", "m"],
                         "valor": [0.620, 0.500, 0.150],
                         "u_tipica": [0.015, 0.0010, 0.0015]})


CONSTRUCTORES = {
    "cinetica_monod.csv": _cinetica_monod,
    "secado_maiz_calibracion.csv": _secado_calibracion,
    "secado_maiz_validacion.csv": _secado_validacion,
    "caudal_mensual.csv": _caudal_mensual,
    "arreglo_fotovoltaico.csv": _arreglo_fotovoltaico,
    "entradas_vertedero.csv": _entradas_vertedero,
}

CARPETA_DATOS = carpeta_datos()


def leer_datos(nombre: str) -> pd.DataFrame:
    """Lee un archivo de datos/ y lo reconstruye si no está presente."""
    ruta = CARPETA_DATOS / nombre
    if not ruta.exists():
        CONSTRUCTORES[nombre]().to_csv(ruta, index=False)
    return pd.read_csv(ruta)


print("Carpeta de datos:", CARPETA_DATOS.name)

## 1. Las dos naturalezas de la incertidumbre

La Definición 4.8 del libro llama aleatoria a la que proviene de la
variabilidad intrínseca del sistema o de sus entradas, y que no
disminuye al acumular más mediciones. La Definición 4.9 llama
epistémica a la que proviene del desconocimiento del modelador, y que
sí disminuye con más información, mejor modelo o malla más fina. La
Figura 4.8 organiza ambas naturalezas frente a los cuatro orígenes
del capítulo.

La distinción gobierna el presupuesto del proyecto. Si la
incertidumbre dominante es epistémica y reside en una constante
cinética, una campaña de ensayos la reduce; si es aleatoria y reside
en la composición del sustrato que llega a planta, lo que corresponde
es dimensionar con margen.

### 1.1 Promediar antes de entrar al modelo es un error caro

El libro advierte que un modelo no lineal evaluado en la media de las
entradas no devuelve la media de las respuestas. La celda siguiente
lo mide sobre la ecuación del vertedero del Ejemplo 4.6, donde la
carga entra elevada a la potencia tres medios.

In [ ]:
G_GRAVEDAD = 9.81   # m/s2


def caudal_vertedero(cd, b, h):
    """Ecuación 4.13 del libro, caudal en m3/s."""
    return (2.0 / 3.0) * cd * b * np.sqrt(2.0 * G_GRAVEDAD) * h**1.5


carga_media, dispersion = 0.150, 0.030      # m, variabilidad real
cargas = rng.normal(carga_media, dispersion, 200_000)
cargas = cargas[cargas > 0.0]

q_de_la_media = caudal_vertedero(0.620, 0.500, carga_media)
media_de_los_q = caudal_vertedero(0.620, 0.500, cargas).mean()

print(f"Caudal evaluado en la carga media   {1e3 * q_de_la_media:.2f} L/s")
print(f"Media de los caudales               {1e3 * media_de_los_q:.2f} L/s")
print(f"Diferencia relativa                 "
      f"{100 * (media_de_los_q - q_de_la_media) / q_de_la_media:.2f} por ciento")
print("\nLa diferencia no es ruido de muestreo, es la curvatura de la")
print("potencia tres medios. Con una variabilidad mayor crecería más.")
assert media_de_los_q > q_de_la_media

## 2. Ley de propagación de primer orden y presupuesto

El Teorema 4.4 del libro recoge la ley de la guía internacional para
la expresión de la incertidumbre en la medida. Cuando la función es
un producto de potencias, la expresión se reduce a la forma relativa,
en la que el exponente de cada variable actúa como amplificador de su
incertidumbre relativa.

El Ejemplo 4.6 mide el caudal de un canal de riego con un vertedero
rectangular de pared delgada sin contracciones. Los datos están en
`datos/entradas_vertedero.csv`.

In [ ]:
entradas = leer_datos("entradas_vertedero.csv")
print(entradas.to_string(index=False))

nominales = entradas["valor"].to_numpy()
tipicas = entradas["u_tipica"].to_numpy()


def caudal(p):
    """Vertedero con las tres entradas en un solo vector."""
    return (2.0 / 3.0) * p[0] * p[1] * np.sqrt(2.0 * G_GRAVEDAD) * p[2]**1.5


def propagacion_primer_orden(f, x, u, rel=1.0e-6):
    """Ley del GUM con derivadas por diferencias, Listado 4.9."""
    x, u = np.asarray(x, float), np.asarray(u, float)
    c = np.zeros_like(x)
    for i in range(x.size):
        d = rel * max(abs(x[i]), 1.0e-12)
        mas, menos = x.copy(), x.copy()
        mas[i], menos[i] = mas[i] + d, menos[i] - d
        c[i] = (f(mas) - f(menos)) / (2 * d)
    contrib = (c * u) ** 2
    return f(x), np.sqrt(contrib.sum()), contrib / contrib.sum(), c


Q_nominal, u_combinada, aporte, sensibilidades = propagacion_primer_orden(
    caudal, nominales, tipicas)

print(f"\nCaudal nominal {1e3 * Q_nominal:.2f} L/s   (libro 53.18 L/s)")
print(f"Incertidumbre combinada {1e3 * u_combinada:.3f} L/s "
      f"(libro 1.518 L/s)")
print(f"Incertidumbre relativa {100 * u_combinada / Q_nominal:.3f} "
      f"por ciento   (libro 2.854 por ciento)")
print(f"Incertidumbre expandida con k = 2: "
      f"{2e3 * u_combinada:.2f} L/s   (libro 3.04 L/s)")
assert abs(1e3 * Q_nominal - 53.18) < 5.0e-3
assert abs(1e3 * u_combinada - 1.518) < 5.0e-4
assert abs(100 * u_combinada / Q_nominal - 2.854) < 5.0e-4

### 2.1 La Tabla 4.5, el presupuesto de incertidumbre

La Definición 4.10 llama presupuesto a la tabla que reúne, para cada
magnitud de entrada, su valor nominal, su incertidumbre típica, su
coeficiente de sensibilidad, su contribución y la fracción que esa
contribución representa del total. La utilidad está en la columna de
fracciones, que ordena las fuentes y señala cuál conviene atacar.

In [ ]:
presupuesto = pd.DataFrame({
    "magnitud": entradas["magnitud"],
    "valor": nominales,
    "u_x": tipicas,
    "c_i": sensibilidades,
    "c_i_u_i_L_s": 1e3 * sensibilidades * tipicas,
    "aporte_pct": 100 * aporte})
print(presupuesto.to_string(index=False,
                            float_format=lambda v: f"{v:.4f}"))
print(f"\nu_c(Q) = {1e3 * u_combinada:.3f} L/s con aporte total "
      f"{100 * aporte.sum():.1f} por ciento")

# Cifras de la Tabla 4.5 con la tolerancia que impone el número de
# decimales con que cada columna se publica.
LIBRO_TABLA_4_5 = {"c_i": ([0.0858, 0.1064, 0.5318], 1.0e-4),
                   "c_i_u_i_L_s": ([1.287, 0.106, 0.798], 1.0e-3),
                   "aporte_pct": ([71.9, 0.5, 27.6], 1.0e-1)}
for columna, (publicado, tolerancia) in LIBRO_TABLA_4_5.items():
    assert np.allclose(presupuesto[columna], publicado,
                       atol=tolerancia, rtol=0.0), columna
print("La Tabla 4.5 del libro se reproduce en sus tres columnas.")

# Forma relativa del Teorema 4.4, con el exponente como amplificador.
exponentes = np.array([1.0, 1.0, 1.5])
relativas = exponentes * tipicas / nominales
for nombre, valor in zip(entradas["magnitud"], relativas):
    print(f"aporte relativo de {nombre}: {100 * valor:.2f} por ciento")
print(f"combinación cuadrática {100 * np.sqrt(np.sum(relativas**2)):.3f} "
      f"por ciento")
assert np.allclose(100 * relativas, [2.42, 0.20, 1.50], atol=5.0e-3)

## 3. Propagación de distribuciones por Monte Carlo

Cuando la linealidad local no se sostiene, el suplemento de la misma
guía propone propagar distribuciones en lugar de varianzas. El
Algoritmo 4.4 del libro fija el criterio de decisión entre ambos
métodos, y declara insuficiente la linealización cuando la desviación
de Monte Carlo difiere de la combinada en más del cinco por ciento.

El Listado 4.10 muestrea con entradas normales independientes y
devuelve el intervalo de cobertura por percentiles.

In [ ]:
def propagacion_monte_carlo(f, x, u, n=200_000, semilla=SEMILLA):
    """Propagación con entradas normales independientes, Listado 4.10."""
    generador = np.random.default_rng(semilla)
    muestras = generador.normal(np.asarray(x, float),
                                np.asarray(u, float), (n, np.size(x)))
    y = np.array([f(fila) for fila in muestras])
    inf, sup = np.percentile(y, [2.5, 97.5])
    return y.mean(), y.std(ddof=1), (inf, sup), y


media_mc, desv_mc, cobertura, muestras_q = propagacion_monte_carlo(
    caudal, nominales, tipicas)
lineal = (Q_nominal - 1.96 * u_combinada, Q_nominal + 1.96 * u_combinada)
discrepancia = 100 * (desv_mc - u_combinada) / u_combinada

print(f"Monte Carlo, desviación {1e3 * desv_mc:.2f} L/s   (libro 1.52 L/s)")
print(f"Monte Carlo, cobertura  {1e3 * cobertura[0]:.2f} a "
      f"{1e3 * cobertura[1]:.2f} L/s   (libro 50.22 a 56.20 L/s)")
print(f"Fórmula lineal          {1e3 * lineal[0]:.2f} a "
      f"{1e3 * lineal[1]:.2f} L/s   (libro 50.21 a 56.16 L/s)")
print(f"Discrepancia entre métodos {discrepancia:.1f} por ciento   "
      f"(libro 0.2 por ciento)")
assert abs(1e3 * desv_mc - 1.52) < 5.0e-3
assert abs(1e3 * cobertura[0] - 50.22) < 5.0e-3
assert abs(1e3 * cobertura[1] - 56.20) < 5.0e-3
assert abs(1e3 * lineal[0] - 50.21) < 5.0e-3
assert abs(1e3 * lineal[1] - 56.16) < 5.0e-3
assert abs(discrepancia) < 5.0
print("\nLa linealización es adecuada, según el criterio del Algoritmo 4.4.")

### 3.1 Estabilidad del muestreo

El Algoritmo 4.4 pide repetir con otra semilla y con el doble de
muestras para verificar la estabilidad. Sin esa comprobación, un
resultado de Monte Carlo no es reportable.

In [ ]:
for semilla_prueba, n_muestras in ((SEMILLA, 200_000),
                                   (SEMILLA + 7, 200_000),
                                   (SEMILLA, 400_000)):
    _, desv, cob, _ = propagacion_monte_carlo(
        caudal, nominales, tipicas, n=n_muestras, semilla=semilla_prueba)
    print(f"semilla {semilla_prueba}, {n_muestras:7d} muestras -> "
          f"desviación {1e3 * desv:.3f} L/s, cobertura "
          f"{1e3 * cob[0]:.2f} a {1e3 * cob[1]:.2f} L/s")
print("\nLas tres corridas coinciden en dos cifras, de modo que el")
print("muestreo es suficiente para el reporte.")

### 3.2 Hasta dónde vale la linealización

La adecuación de la linealización no es propiedad del método sino del
modelo. La Figura 4.9 del libro compara ambos procedimientos sobre el
vertedero y explora el límite de validez para tres formas
funcionales. Una potencia moderada tolera incertidumbres relativas
grandes, un cociente empieza a fallar cerca del 15 por ciento y una
exponencial de argumento inverso, como la de toda cinética de
Arrhenius, falla antes de lo que la intuición sugiere.

In [ ]:
from scipy import stats

fig, (izq, der) = plt.subplots(1, 2, figsize=(10.5, 4.3))

izq.hist(muestras_q * 1e3, bins=90, density=True, color=PALETA["azul"],
         alpha=0.55, edgecolor="none", label="Monte Carlo")
malla_q = np.linspace((Q_nominal - 4.2 * u_combinada) * 1e3,
                      (Q_nominal + 4.2 * u_combinada) * 1e3, 500)
izq.plot(malla_q, stats.norm.pdf(malla_q, Q_nominal * 1e3,
                                 u_combinada * 1e3),
         "-", color=PALETA["rojo"], lw=1.3, label="primer orden")
for extremo in lineal:
    izq.axvline(extremo * 1e3, color=PALETA["rojo"], ls="--", lw=0.9)
for extremo in cobertura:
    izq.axvline(extremo * 1e3, color=PALETA["azul"], ls=":", lw=1.1)
izq.set_xlabel("Caudal Q (L/s)")
izq.set_ylabel("Densidad (s/L)")
izq.set_title("(a) vertedero, Q proporcional a h elevado a 1.5")
izq.legend(loc="upper right", fontsize=8)

CASOS = (
    ("potencia tres medios", lambda x: x**1.5,
     lambda x: 1.5 * x**0.5, 0.15, PALETA["verde"], "o"),
    ("cociente inverso", lambda x: 1.0 / x,
     lambda x: -1.0 / x**2, 0.030, PALETA["naranja"], "s"),
    ("exponencial de argumento inverso", lambda x: np.exp(1.0e4 / x),
     lambda x: -1.0e4 / x**2 * np.exp(1.0e4 / x), 298.0,
     PALETA["morado"], "^"),
)
generador_casos = np.random.default_rng(SEMILLA)
for etiqueta, f, df, x0, color, marcador in CASOS:
    elasticidad = abs(df(x0) * x0 / f(x0))
    relativa, discrepancias = [], []
    for objetivo in np.linspace(0.01, 0.50, 17):
        u = objetivo * x0 / elasticidad
        if u / x0 > 0.30:
            break
        u_lineal = abs(df(x0)) * u
        y = f(stats.truncnorm.rvs((0.0 - x0) / u, np.inf, loc=x0,
                                  scale=u, size=60_000,
                                  random_state=generador_casos))
        relativa.append(100 * u_lineal / abs(f(x0)))
        discrepancias.append(100 * (y.std(ddof=1) - u_lineal) / u_lineal)
    der.plot(relativa, discrepancias, marcador + "-", color=color,
             ms=3.2, label=etiqueta)
der.axhline(0.0, color=PALETA["gris"], lw=0.9)
der.axhline(5.0, color=PALETA["gris"], ls="--", lw=0.9)
der.set_xlabel("Incertidumbre de primer orden (por ciento)")
der.set_ylabel("Discrepancia frente a Monte Carlo (por ciento)")
der.set_title("(b) validez de la linealización")
der.set_xlim(0.0, 52.0)
der.set_ylim(-4.0, 45.0)
der.legend(loc="upper left", fontsize=7.5)

fig.tight_layout()
plt.show()

## 4. Consistencia antes de propagar

La sección 4.4.2 del libro enumera cuatro comprobaciones baratas que
atrapan la mayoría de los errores gruesos, y son la homogeneidad
dimensional de toda ecuación implementada, el cierre de los balances,
la comparación del orden de magnitud con un cálculo independiente y
el contraste con valores publicados.

### 4.1 Homogeneidad dimensional con álgebra simbólica

SymPy permite comprobar la homogeneidad sin ejecutar el modelo, sobre
la propia expresión. Basta sustituir cada símbolo por su dimensión y
verificar que los dos lados coinciden.

In [ ]:
import sympy as sp

LONGITUD, TIEMPO = sp.symbols("L T", positive=True)
cd_s, b_s, h_s, g_s = sp.symbols("C_d b h g", positive=True)
DIMENSIONES = {cd_s: sp.Integer(1), b_s: LONGITUD, h_s: LONGITUD,
               g_s: LONGITUD / TIEMPO**2}


def es_homogenea(expresion, esperada, dimensiones) -> bool:
    """Verdadero si la razón entre la expresión y lo esperado es un número."""
    razon = sp.simplify(expresion.subs(dimensiones) / esperada)
    return bool(razon.is_number)


expresion_q = (sp.Rational(2, 3) * cd_s * b_s * sp.sqrt(2 * g_s)
               * h_s**sp.Rational(3, 2))
dimension_q = sp.simplify(expresion_q.subs(DIMENSIONES))
esperada = LONGITUD**3 / TIEMPO

print("Dimensión de la Ecuación 4.13:", dimension_q)
print("Dimensión esperada del caudal:", esperada)
print("Razón entre ambas:", sp.simplify(dimension_q / esperada))
assert es_homogenea(expresion_q, esperada, DIMENSIONES)
print("La Ecuación 4.13 es dimensionalmente homogénea, porque la razón")
print("entre su dimensión y la del caudal es un número sin símbolos.")

# Un término mal escrito se detecta de inmediato.
expresion_mala = expresion_q + b_s * h_s
print("\nCon un término espurio la razón deja de ser un número:")
print("  ", sp.simplify(expresion_mala.subs(DIMENSIONES) / esperada))
assert not es_homogenea(expresion_mala, esperada, DIMENSIONES)

### 4.2 Cierre de balances y orden de magnitud

La segunda comprobación exige que lo que entra menos lo que sale
iguale lo que se acumula, con una tolerancia declarada. La tercera
compara el resultado con una estimación independiente hecha con la
aritmética más burda posible.

In [ ]:
def cierre_de_balance(entra, sale, acumula, tolerancia=1.0e-9):
    """Residuo del balance relativo al mayor flujo, adimensional."""
    escala = max(abs(entra), abs(sale), abs(acumula), 1.0e-30)
    residuo = abs(entra - sale - acumula) / escala
    return residuo, residuo <= tolerancia


# Balance de agua en un tramo de canal durante una hora, en m3.
entrada = Q_nominal * 3600.0
salida = 0.95 * entrada
almacenado = entrada - salida
residuo, cierra = cierre_de_balance(entrada, salida, almacenado)
print(f"Entra {entrada:.2f} m3, sale {salida:.2f} m3, "
      f"almacena {almacenado:.2f} m3")
print(f"Residuo relativo {residuo:.2e}, cierra: {cierra}")
assert cierra

# Orden de magnitud con la aritmética más burda posible.
burdo = 0.7 * 0.5 * 4.4 * 0.15**1.5   # dos tercios por raíz de 2g
print(f"\nEstimación burda {1e3 * burdo:.0f} L/s frente al cálculo "
      f"completo {1e3 * Q_nominal:.0f} L/s")
assert 0.5 < burdo / Q_nominal < 2.0
print("Coinciden en orden de magnitud, que es lo que esta comprobación pide.")

### 4.3 Cifras significativas gobernadas por la incertidumbre

El libro fija una regla sencilla que casi nadie aplica. La
incertidumbre se redondea a una o dos cifras significativas y el
valor a la misma posición decimal, de modo que escribir 53.1810 L/s
con incertidumbre de 3 L/s es una contradicción interna. Automatizar
ese redondeo en el código que produce las tablas evita que un solo
número del informe la incumpla.

In [ ]:
def posicion_significativa(u: float, cifras: int = 2) -> int:
    """Posición decimal a la que redondear, positiva hacia la derecha."""
    if u == 0.0 or not np.isfinite(u):
        return 0
    return int(cifras - 1 - np.floor(np.log10(abs(u))))


def redondear_con_incertidumbre(valor: float, u: float,
                                cifras: int = 2) -> tuple[float, float]:
    """Redondea la incertidumbre a cifras significativas y el valor igual."""
    posicion = posicion_significativa(u, cifras)
    return round(valor, posicion), round(u, posicion)


def formato_valor_incertidumbre(valor: float, u: float,
                                cifras: int = 2, unidad: str = "") -> str:
    """Texto del tipo 53.18 +- 1.52 L/s, con decimales coherentes."""
    v, uu = redondear_con_incertidumbre(valor, u, cifras)
    decimales = max(posicion_significativa(u, cifras), 0)
    sufijo = f" {unidad}" if unidad else ""
    return f"{v:.{decimales}f} +- {uu:.{decimales}f}{sufijo}"


tipica = formato_valor_incertidumbre(1e3 * Q_nominal,
                                     1e3 * u_combinada, 2, "L/s")
expandida = formato_valor_incertidumbre(1e3 * Q_nominal,
                                        2e3 * u_combinada, 2, "L/s")
una_cifra = formato_valor_incertidumbre(1e3 * Q_nominal,
                                        2e3 * u_combinada, 1, "L/s")

print("Caudal del Ejemplo 4.6 con incertidumbre típica:  ", tipica)
print("Con incertidumbre expandida, factor de cobertura 2:", expandida)
print("El libro lo reporta como 53.2 con incertidumbre 3.0 L/s.")
print("Con una sola cifra en la incertidumbre quedaría:   ", una_cifra)
assert expandida == "53.2 +- 3.0 L/s"
assert una_cifra == "53 +- 3 L/s"

print("\nEl reporte de 53.1810 L/s con incertidumbre de 3 L/s que el")
print("libro señala como contradicción interna queda así corregido.")

### 4.4 Las cinco limitaciones que deben declararse siempre

El libro cierra la sección con una obligación profesional. Cinco
limitaciones deben aparecer siempre por escrito, y omitirlas
transfiere al lector un riesgo que no le corresponde. La celda
siguiente las deja como estructura de datos, para que el informe las
arrastre y ninguna se pierda.

In [ ]:
LIMITACIONES_OBLIGATORIAS = [
    "dominio de validez cubierto por los datos de validación",
    "procesos omitidos del modelo",
    "fuentes de incertidumbre no cuantificadas",
    "dependencia respecto de escenarios futuros no verificables",
    "error numérico residual",
]

limitaciones_vertedero = {
    "dominio de validez cubierto por los datos de validación":
        "cargas entre 0.05 m y 0.25 m sobre la cresta",
    "procesos omitidos del modelo":
        "contracciones laterales y arrastre de sedimento",
    "fuentes de incertidumbre no cuantificadas":
        "deriva del limnímetro entre calibraciones",
    "dependencia respecto de escenarios futuros no verificables":
        "ninguna, la medición es presente",
    "error numérico residual":
        "despreciable, la ecuación es explícita",
}

faltantes = [t for t in LIMITACIONES_OBLIGATORIAS
             if t not in limitaciones_vertedero]
for titulo in LIMITACIONES_OBLIGATORIAS:
    print(f"- {titulo}: {limitaciones_vertedero.get(titulo, 'SIN DECLARAR')}")
assert not faltantes, faltantes
print("\nLas cinco limitaciones están declaradas.")

## 5. Ejercicios guiados

Las celdas siguientes llevan la marca `# COMPLETE:` y arrancan con un
valor de partida evidentemente incorrecto, de modo que el cuaderno
corre completo aunque falten por resolver. Al terminarlas, cambie
`REVISAR = True` en la celda de configuración.

### Ejercicio 1. Problema 4-19, presupuesto de una bomba

La potencia hidráulica de una bomba es el producto de densidad,
gravedad, caudal y altura dividido por la eficiencia. Con
incertidumbres relativas del 0.1 por ciento, 3 por ciento, 2 por
ciento y 4 por ciento, construya su presupuesto. Recuerde la forma
relativa del Teorema 4.4, donde el exponente de cada variable
amplifica su incertidumbre relativa, y note que la eficiencia entra
con exponente menos uno.

In [ ]:
relativas_bomba = np.array([0.001, 0.03, 0.02, 0.04])  # rho, Q, H, eta
exponentes_bomba = np.array([1.0, 1.0, 1.0, -1.0])

# COMPLETE: calcule la incertidumbre relativa combinada de la
# potencia y el aporte de cada fuente a la varianza. El aporte es la
# contribución de cada término dividida por la suma de todas.
u_rel_bomba = None
aportes_bomba = None

In [ ]:
# Verificación del ejercicio 1.
verificar("incertidumbre relativa de la potencia", u_rel_bomba, 0.053861, tol=1.0e-4)
if aportes_bomba is not None:
    etiquetas = ["densidad", "caudal", "altura", "eficiencia"]
    for nombre, aporte_i in zip(etiquetas, aportes_bomba):
        print(f"{nombre:12s} aporta {100 * aporte_i:5.1f} por ciento "
              f"de la varianza")
    verificar("aporte de la eficiencia", aportes_bomba[3], 0.551534, tol=1.0e-4)
    print("\nAtacar la eficiencia rinde más que medir la densidad mejor.")

### Ejercicio 2. Problema 4-20, pérdida de carga

La pérdida de carga de una tubería depende del cuadrado del caudal.
Con incertidumbre relativa del 6 por ciento en el caudal, calcule la
de la pérdida por la ley de primer orden y verifíquela por Monte
Carlo con 100000 réplicas y la semilla del curso.

In [ ]:
caudal_nominal, u_rel_caudal = 0.050, 0.06     # m3/s y fracción


def perdida(q):
    """Pérdida de carga proporcional al cuadrado del caudal, en m."""
    return 4.2e3 * q**2


# COMPLETE: calcule la incertidumbre relativa de la pérdida por la
# ley de primer orden, y después por Monte Carlo con 100000 réplicas
# y semilla SEMILLA.
u_rel_perdida_lineal = None
u_rel_perdida_mc = None

In [ ]:
# Verificación del ejercicio 2.
verificar("incertidumbre relativa por primer orden", u_rel_perdida_lineal,
          0.12, tol=1.0e-3)
verificar("incertidumbre relativa por Monte Carlo", u_rel_perdida_mc,
          0.120087, tol=1.0e-3)
if u_rel_perdida_mc is not None:
    exceso = 100 * (u_rel_perdida_mc - 0.12) / 0.12
    print(f"\nMonte Carlo difiere de la ley lineal en {exceso:.2f} por ciento,")
    print("muy por debajo del cinco por ciento del Algoritmo 4.4, de modo")
    print("que la linealización es adecuada con este nivel de ruido.")

### Ejercicio 3. Problema 4-21, la exponencial de Arrhenius

La vida útil por Arrhenius depende de la temperatura a través de una
exponencial de argumento inverso. Compruebe por qué la ley de primer
orden falla antes que en un producto de potencias, midiendo la
discrepancia con Monte Carlo para una incertidumbre de 5 K en una
temperatura de 298 K.

In [ ]:
B_ARRHENIUS, T_NOMINAL, U_TEMPERATURA = 1.0e4, 298.0, 5.0   # K, K, K


def vida_util(temperatura):
    """Vida útil relativa por Arrhenius, adimensional."""
    return np.exp(B_ARRHENIUS / temperatura)


# COMPLETE: calcule la incertidumbre por primer orden usando la
# derivada analítica de la vida útil, y la desviación por Monte Carlo
# con 200000 réplicas y semilla SEMILLA. Guarde la razón entre ambas
# en razon_arrhenius.
u_lineal_arrhenius = None
u_mc_arrhenius = None
razon_arrhenius = None

In [ ]:
# Verificación del ejercicio 3.
verificar("razón entre Monte Carlo y primer orden", razon_arrhenius,
          1.3311, tol=2.0e-3)
if razon_arrhenius is not None:
    print(f"\nLa desviación de Monte Carlo supera a la lineal en un factor")
    print(f"{razon_arrhenius:.2f}, muy por encima del cinco por ciento que")
    print("el Algoritmo 4.4 admite. El argumento inverso dentro de una")
    print("exponencial amplifica la curvatura y rompe la linealización.")

### Ejercicio 4. Problema 4-22, el reporte mal redondeado

Un resultado se reporta como 1247.836 kWh con incertidumbre de 63
kWh. Corrija el reporte con la función de la sección 4.3 y justifique
la regla aplicada.

In [ ]:
# COMPLETE: use formato_valor_incertidumbre con dos cifras
# significativas en la incertidumbre y guarde el texto corregido en
# reporte_corregido.
reporte_corregido = None

In [ ]:
# Verificación del ejercicio 4.
print("Reporte original  1247.836 +- 63 kWh")
print("Reporte corregido", reporte_corregido)
if REVISAR:
    assert reporte_corregido == "1248 +- 63 kWh", reporte_corregido
elif reporte_corregido is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    estado = "ok" if reporte_corregido == "1248 +- 63 kWh" else "revisar"
    print(f"[{estado}] se esperaba 1248 +- 63 kWh")
print("\nLos tres decimales del original prometen una precisión de un")
print("gramo de kWh que la incertidumbre de 63 kWh desmiente por completo.")

### Ejercicio 5. Problema 4-23, la diferencia entre dos escenarios

La incertidumbre de la diferencia entre dos escenarios evaluados con
el mismo modelo es menor que la de cada uno por separado, porque
buena parte de la incertidumbre es común y se cancela. Compruébelo
para dos escenarios con incertidumbre típica de 460 kWh cada uno y
correlación de 0.95 entre ellos.

In [ ]:
u_escenario, correlacion_escenarios = 460.0, 0.95   # kWh

# COMPLETE: calcule la incertidumbre de la diferencia entre los dos
# escenarios, que para dos magnitudes correlacionadas vale la raíz de
# la suma de varianzas menos dos veces la covarianza.
u_diferencia = None

In [ ]:
# Verificación del ejercicio 5.
verificar("incertidumbre de la diferencia", u_diferencia, 145.4648, tol=1.0e-4)
if u_diferencia is not None:
    print(f"\nCada escenario carga {u_escenario:.0f} kWh y la diferencia")
    print(f"apenas {u_diferencia:.0f} kWh. La cancelación desaparece cuando")
    print("la correlación es nula, y se invierte si es negativa.")
    sin_correlacion = np.sqrt(2) * u_escenario
    print(f"Sin correlación la diferencia cargaría {sin_correlacion:.0f} kWh.")

## 6. Problemas del capítulo

### Problema 4-3, andamiaje

Distinga incertidumbre aleatoria de epistémica sobre tres magnitudes
de su área, e indique para cada una si más mediciones la reducirían.
La estructura siguiente le sirve de plantilla, y el cuaderno verifica
que la clasificación sea coherente con las Definiciones 4.8 y 4.9.

In [ ]:
MAGNITUDES = pd.DataFrame([
    {"magnitud": "humedad inicial del lote de grano",
     "naturaleza": "aleatoria",
     "vía de reducción": "dimensionar con margen, no medir más"},
    {"magnitud": "constante cinética de secado",
     "naturaleza": "epistémica",
     "vía de reducción": "campaña de ensayos adicionales"},
    {"magnitud": "error de discretización del integrador",
     "naturaleza": "epistémica",
     "vía de reducción": "malla más fina o esquema de mayor orden"},
])
MAGNITUDES["reducible_midiendo"] = (
    MAGNITUDES["naturaleza"] == "epistémica")
print(MAGNITUDES.to_string(index=False))
assert MAGNITUDES["naturaleza"].isin(["aleatoria", "epistémica"]).all()
print("\nSolo lo epistémico se reduce con más información, según la")
print("Definición 4.9. Lo aleatorio describe una dispersión real.")

### Problema 4-24, resuelto en parte

Implemente una función que reciba una lista de mallas, estime el
orden observado y avise cuando el régimen asintótico no se haya
alcanzado. La versión de la sección 3 del cuaderno U4_01 ya lo hace;
aquí se comprueba que el aviso funcione sobre una sucesión que no ha
llegado al régimen.

In [ ]:
def orden_con_aviso(h, e, tolerancia=0.05):
    """Orden por pares, pendiente global y aviso de régimen asintótico."""
    h, e = np.asarray(h, float), np.asarray(e, float)
    p_par = np.log(e[:-1] / e[1:]) / np.log(h[:-1] / h[1:])
    p_reg = float(np.polyfit(np.log(h), np.log(e), 1)[0])
    asintotico = bool(abs(p_par[-1] - p_par[-2]) < tolerancia)
    return p_par, p_reg, asintotico


sucesion_buena = (np.array([0.04, 0.02, 0.01, 0.005]),
                  np.array([4.10e-3, 1.05e-3, 2.64e-4, 6.60e-5]))
sucesion_mala = (np.array([0.04, 0.02, 0.01, 0.005]),
                 np.array([4.10e-3, 1.60e-3, 6.00e-4, 2.10e-4]))
for etiqueta, (h_s, e_s) in (("en régimen", sucesion_buena),
                             ("fuera de régimen", sucesion_mala)):
    p_par_s, p_reg_s, aviso = orden_con_aviso(h_s, e_s)
    print(f"{etiqueta:18s} pares {np.round(p_par_s, 3)} "
          f"global {p_reg_s:.3f} asintótico {aviso}")
assert orden_con_aviso(*sucesion_buena)[2]
assert not orden_con_aviso(*sucesion_mala)[2]

## Cierre

### Lo que debe saber hacer al terminar

- Separar la incertidumbre aleatoria de la epistémica y decir cuál se reduce midiendo más.
- Calcular coeficientes de sensibilidad por diferencias centradas y armar el presupuesto de incertidumbre ordenado por aporte.
- Propagar por Monte Carlo, verificar la estabilidad del muestreo y decidir si la linealización basta.
- Comprobar homogeneidad dimensional, cierre de balances y orden de magnitud antes de dar por bueno un resultado.
- Redondear un resultado con la incertidumbre gobernando las cifras significativas, y escribir las cinco limitaciones obligatorias.

### Qué revisar en el libro si algo no salió

- Si el presupuesto no coincide, revise el Teorema 4.4, el Listado 4.9 y la Tabla 4.5.
- Si Monte Carlo y la ley de primer orden discrepan, revise el Algoritmo 4.4 y la Figura 4.9.
- Si duda del redondeo, revise la sección 4.4.2, donde la regla de las cifras significativas está enunciada.
- Si no sabe qué limitaciones declarar, la sección 4.4.2 enumera las cinco que deben aparecer siempre por escrito.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente de programación.
Todo resultado numérico que aparece aquí se verifica contra el valor
que el libro publica, contra una solución analítica o contra un caso
límite, según recuerda la sección 4.6 del libro. La responsabilidad
del contenido no se transfiere al asistente.